<div align="center">

# Predicción de tasas de mortalidad y natalidad a partir de indicadores socioeconómicos

**Proyecto Integrador**

Universidad Internacional del Ecuador — UIDE

---

**Notebook 00 — Configuración del entorno de desarrollo**

---

Equipo: Guillermo Paredes · Donato Oña · Mateo Villacreses

Junio 2026

</div>

## Descripción

Este notebook define el entorno de desarrollo común sobre el cual se ejecutan todos los notebooks del proyecto. La configuración se materializa como un **devcontainer** ejecutado por GitHub Codespaces, lo que garantiza que cualquier integrante del equipo trabaje sobre exactamente la misma versión de Python, las mismas librerías y las mismas extensiones del editor.

### Stack tecnológico

| Componente | Detalle |
|---|---|
| Python | 3.13 (imagen oficial `mcr.microsoft.com/devcontainers/python`) |
| Cómputo numérico | NumPy, Pandas, SciPy |
| Visualización | Matplotlib, Seaborn, Plotly |
| ML clásico | scikit-learn, XGBoost, LightGBM, statsmodels |
| Deep Learning | PyTorch (build CPU) |
| Fuente de datos | wbgapi (cliente oficial del Banco Mundial) |
| Entorno interactivo | Jupyter integrado en VS Code |

### Estructura del proyecto

```
proyecto/
├── .devcontainer/
│   └── devcontainer.json
├── notebooks/
│   ├── 00_setup.ipynb
│   └── 01_eda.ipynb
├── src/
├── data/
├── requirements.txt
└── .gitignore
```

## 1. Generación de la estructura de directorios

In [1]:
import os

for carpeta in [".devcontainer", "notebooks", "data", "src"]:
    os.makedirs(carpeta, exist_ok=True)

# .gitkeep para que git tracking las carpetas vacías de trabajo
for carpeta in ["notebooks", "src"]:
    open(os.path.join(carpeta, ".gitkeep"), "a").close()

print("✅ Estructura creada")
!ls -la

✅ Estructura creada
total 40
drwxrwxrwx+ 7 vscode root   4096 Jun  6 01:52 .
drwxr-xrwx+ 6 vscode root   4096 Jun  6 01:43 ..
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 00:45 .devcontainer
drwxrwxrwx+ 8 vscode root   4096 Jun  6 01:14 .git
-rw-rw-rw-  1 vscode vscode  425 Jun  6 00:46 .gitignore
-rw-rw-rw-  1 vscode root     13 Jun  3 22:38 README.md
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 01:43 data
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 01:52 notebooks
-rw-rw-rw-  1 vscode vscode  595 Jun  6 00:45 requirements.txt
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 01:52 src


## 2. Definición del devcontainer

El archivo `.devcontainer/devcontainer.json` especifica la imagen base, las extensiones de VS Code que se instalan automáticamente y el comando de post-creación que aprovisiona las dependencias de Python.

PyTorch se instala desde el índice CPU oficial (`download.pytorch.org/whl/cpu`) para evitar la descarga de los binarios CUDA (~3 GB) que no se utilizan en este proyecto.

In [2]:
%%writefile .devcontainer/devcontainer.json
{
  "name": "Proyecto ML/DS",
  "image": "mcr.microsoft.com/devcontainers/python:1-3.13-bookworm",

  "features": {
    "ghcr.io/devcontainers/features/git:1": {}
  },

  "customizations": {
    "vscode": {
      "extensions": [
        "ms-python.python",
        "ms-python.vscode-pylance",
        "ms-toolsai.jupyter",
        "ms-toolsai.jupyter-keymap",
        "ms-toolsai.jupyter-renderers",
        "ms-toolsai.vscode-jupyter-cell-tags",
        "ms-toolsai.vscode-jupyter-slideshow",
        "charliermarsh.ruff"
      ],
      "settings": {
        "python.defaultInterpreterPath": "/usr/local/bin/python",
        "jupyter.notebookFileRoot": "${workspaceFolder}",
        "[python]": {
          "editor.defaultFormatter": "charliermarsh.ruff",
          "editor.formatOnSave": true
        }
      }
    }
  },

  "postCreateCommand": "pip install --upgrade pip && pip install --index-url https://download.pytorch.org/whl/cpu torch~=2.5 && pip install -r requirements.txt",

  "remoteUser": "vscode"
}


Overwriting .devcontainer/devcontainer.json


## 3. Dependencias del proyecto

Las versiones se fijan con el operador `~=` (compatible release), lo que permite recibir parches sin saltos mayores de versión.

In [ ]:
%%writefile requirements.txt
# === Núcleo numérico ===
numpy~=2.1
pandas~=2.2
scipy~=1.14

# === Visualización ===
matplotlib~=3.9
seaborn~=0.13
plotly~=5.24

# === ML clásico ===
scikit-learn~=1.5
xgboost~=2.1
lightgbm~=4.5
statsmodels~=0.14

# === Interpretabilidad ===
shap~=0.46

# === Deep Learning (CPU) ===
# torch se instala desde el índice CPU vía postCreateCommand del devcontainer
# Descomenta la siguiente línea si el equipo decide usar TensorFlow también
# tensorflow-cpu~=2.18

# === Jupyter (kernel; la UI viene en VS Code vía extensiones) ===
ipykernel~=6.29
ipywidgets~=8.1

# === Utilidades ===
tqdm~=4.66
joblib~=1.4
python-dotenv~=1.0

# === API y datos (Banco Mundial + Parquet) ===
wbgapi==1.0.12
pyarrow~=17.0


## 4. Configuración del control de versiones

El archivo `.gitignore` excluye del repositorio los datos crudos, los modelos serializados, los archivos generados por Python y los artefactos de IDE. En particular, **el directorio `data/` no se versiona**: cada integrante regenera los datos ejecutando el notebook de EDA, lo que evita commitear archivos pesados y mantiene la reproducibilidad mediante código en lugar de archivos.

In [ ]:
%%writefile .gitignore
# Datos y modelos
data/*
!data/.gitkeep
models/
*.csv
*.parquet
*.h5
*.pkl
*.pt
*.pth
*.ckpt

# Python
__pycache__/
*.py[cod]
*$py.class
*.egg-info/
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Jupyter
.ipynb_checkpoints/

# Entornos virtuales (por si alguien trabaja en local también)
.venv/
venv/
env/

# Secretos
.env
*.key

# IDE
.vscode/*
!.vscode/settings.json
!.vscode/extensions.json
.idea/

# SO
.DS_Store
Thumbs.db


Overwriting .gitignore


In [ ]:
# Marcador para conservar el directorio data/ en el repositorio aunque esté vacío
open("data/.gitkeep", "a").close()
print("✅ data/.gitkeep creado")

✅ data/.gitkeep creado


## 5. Verificación de los archivos generados

In [6]:
!ls -la && echo '---' && ls -la .devcontainer/

total 40
drwxrwxrwx+ 7 vscode root   4096 Jun  6 01:52 .
drwxr-xrwx+ 6 vscode root   4096 Jun  6 01:43 ..
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 00:45 .devcontainer
drwxrwxrwx+ 8 vscode root   4096 Jun  6 01:14 .git
-rw-rw-rw-  1 vscode vscode  425 Jun  6 01:52 .gitignore
-rw-rw-rw-  1 vscode root     13 Jun  3 22:38 README.md
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 01:52 data
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 01:52 notebooks
-rw-rw-rw-  1 vscode vscode  595 Jun  6 01:52 requirements.txt
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 01:52 src
---
total 12
drwxrwxrwx+ 2 vscode vscode 4096 Jun  6 00:45 .
drwxrwxrwx+ 7 vscode root   4096 Jun  6 01:52 ..
-rw-rw-rw-  1 vscode vscode 1015 Jun  6 01:52 devcontainer.json


## 6. Procedimiento de despliegue

### Inicialización del repositorio

Una vez generados los archivos de configuración, se versionan en el repositorio:

```bash
git add .devcontainer/ requirements.txt .gitignore notebooks/ src/ data/.gitkeep 00_setup.ipynb
git commit -m "feat: configuración inicial del Codespace (devcontainer y dependencias)"
git push
```

Tras el push, el Codespace activo debe reconstruirse para que adopte la nueva configuración: en la paleta de comandos (`Ctrl+Shift+P` o `Cmd+Shift+P`) se ejecuta **Codespaces: Rebuild Container**. El primer build toma aproximadamente 3–5 minutos.

### Incorporación de nuevos integrantes

Cualquier integrante del equipo accede al entorno desde la página del repositorio en GitHub mediante **Code → Codespaces → Create codespace on main**. GitHub aprovisiona el contenedor y ejecuta automáticamente el `postCreateCommand`, dejando listo el entorno con todas las dependencias instaladas.

### Actualización de dependencias

Cuando se incorpora una nueva librería, se añade a `requirements.txt`, se hace commit y push, y cada integrante reconstruye su Codespace mediante **Rebuild Container** para sincronizar el entorno.

## 7. Validación del entorno

Tras la reconstrucción del Codespace, la siguiente celda confirma que todas las librerías críticas están disponibles y reporta sus versiones.

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import seaborn as sns
import plotly
import torch
import xgboost as xgb
import lightgbm as lgb
import wbgapi as wb
import pyarrow

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"SciPy       : {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Seaborn     : {sns.__version__}")
print(f"Plotly      : {plotly.__version__}")
print(f"XGBoost     : {xgb.__version__}")
print(f"LightGBM    : {lgb.__version__}")
print(f"wbgapi      : {wb.__version__}")
print(f"PyArrow     : {pyarrow.__version__}")
print(f"PyTorch     : {torch.__version__}  (CUDA: {torch.cuda.is_available()}, esperado: False)")